In [178]:
# Import

In [179]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

In [180]:
# download datasets: kaggle competitions download -c house-prices-advanced-regression-techniques

In [181]:
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

In [182]:
print(train.shape)
print(train.head())

(1460, 81)
   Id  MSSubClass MSZoning  LotFrontage  LotArea Street Alley LotShape  \
0   1          60       RL         65.0     8450   Pave   NaN      Reg   
1   2          20       RL         80.0     9600   Pave   NaN      Reg   
2   3          60       RL         68.0    11250   Pave   NaN      IR1   
3   4          70       RL         60.0     9550   Pave   NaN      IR1   
4   5          60       RL         84.0    14260   Pave   NaN      IR1   

  LandContour Utilities  ... PoolArea PoolQC Fence MiscFeature MiscVal MoSold  \
0         Lvl    AllPub  ...        0    NaN   NaN         NaN       0      2   
1         Lvl    AllPub  ...        0    NaN   NaN         NaN       0      5   
2         Lvl    AllPub  ...        0    NaN   NaN         NaN       0      9   
3         Lvl    AllPub  ...        0    NaN   NaN         NaN       0      2   
4         Lvl    AllPub  ...        0    NaN   NaN         NaN       0     12   

  YrSold  SaleType  SaleCondition  SalePrice  
0   2008  

In [183]:
# Checking missing values and sorting out

In [184]:
missing = train.isnull().sum()
missing = missing[missing > 0].sort_values(ascending = False)
print(missing)

PoolQC          1453
MiscFeature     1406
Alley           1369
Fence           1179
MasVnrType       872
FireplaceQu      690
LotFrontage      259
GarageType        81
GarageYrBlt       81
GarageFinish      81
GarageQual        81
GarageCond        81
BsmtFinType2      38
BsmtExposure      38
BsmtFinType1      37
BsmtCond          37
BsmtQual          37
MasVnrArea         8
Electrical         1
dtype: int64


In [185]:
cols_to_drop = ['PoolQC', 'MiscFeature', 'Alley', 'Fence', 'MasVnrType', 'FireplaceQu']
train.drop(cols_to_drop, axis=1, inplace=True)

In [186]:
train['LotFrontage'].fillna(train['LotFrontage'].median(), inplace=True)
train['MasVnrArea'].fillna(0, inplace=True)
train['Electrical'].fillna(train['Electrical'].mode()[0], inplace=True)

garage_cols = ['GarageType', 'GarageYrBlt', 'GarageFinish', 'GarageQual', 'GarageCond']
bsmt_cols = ['BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2']

In [187]:
for col in garage_cols + bsmt_cols:
    if train[col].dtype == 'object':
        train[col].fillna('None', inplace=True)
    else:
        train[col].fillna(0, inplace=True)

print(train.isnull().sum())

Id               0
MSSubClass       0
MSZoning         0
LotFrontage      0
LotArea          0
                ..
MoSold           0
YrSold           0
SaleType         0
SaleCondition    0
SalePrice        0
Length: 75, dtype: int64


In [188]:
# converting text column to numbers

In [189]:
cat_cols = train.select_dtypes(include='object').columns
print(cat_cols)
print(len(cat_cols))

Index(['MSZoning', 'Street', 'LotShape', 'LandContour', 'Utilities',
       'LotConfig', 'LandSlope', 'Neighborhood', 'Condition1', 'Condition2',
       'BldgType', 'HouseStyle', 'RoofStyle', 'RoofMatl', 'Exterior1st',
       'Exterior2nd', 'ExterQual', 'ExterCond', 'Foundation', 'BsmtQual',
       'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2', 'Heating',
       'HeatingQC', 'CentralAir', 'Electrical', 'KitchenQual', 'Functional',
       'GarageType', 'GarageFinish', 'GarageQual', 'GarageCond', 'PavedDrive',
       'SaleType', 'SaleCondition'],
      dtype='object')
37


In [190]:
# fitting the model

In [191]:
train_le = train.copy()
le = LabelEncoder()
cat_cols = train.select_dtypes(include='object').columns

for col in cat_cols:
    train_le[col] = le.fit_transform(train_le[col])

In [192]:
y = train_le['SalePrice']
X = train_le.drop('SalePrice', axis=1)

In [193]:
X_train, X_val, y_train, y_val  = train_test_split(X, y, random_state = 23, test_size=0.2)
model = LinearRegression()
model.fit(X_train, y_train)
pred = model.predict(X_val)

rmse = np.sqrt(mean_squared_error(y_val, pred))
r2 = r2_score(y_val, pred)

print('RMSE: ', rmse)
print('R2: ', r2)

RMSE:  25272.874910779
R2:  0.8733198013663855


In [194]:
# Submission And Testing

In [195]:
test_le = test.copy()

In [196]:
missing = test_le.isnull().sum()
missing = missing[missing > 0].sort_values(ascending = False)
print(missing)

PoolQC          1456
MiscFeature     1408
Alley           1352
Fence           1169
MasVnrType       894
FireplaceQu      730
LotFrontage      227
GarageCond        78
GarageYrBlt       78
GarageQual        78
GarageFinish      78
GarageType        76
BsmtCond          45
BsmtExposure      44
BsmtQual          44
BsmtFinType1      42
BsmtFinType2      42
MasVnrArea        15
MSZoning           4
BsmtFullBath       2
BsmtHalfBath       2
Functional         2
Utilities          2
GarageCars         1
GarageArea         1
TotalBsmtSF        1
KitchenQual        1
BsmtUnfSF          1
BsmtFinSF2         1
BsmtFinSF1         1
Exterior2nd        1
Exterior1st        1
SaleType           1
dtype: int64


In [197]:
cols_to_drop = ['PoolQC', 'MiscFeature', 'Alley', 'Fence', 'MasVnrType', 'FireplaceQu']
test_le.drop(cols_to_drop, axis=1, inplace=True)

In [198]:
test_le['LotFrontage'].fillna(test_le['LotFrontage'].median(), inplace=True)
test_le['MasVnrArea'].fillna(0, inplace=True)
test_le['Electrical'].fillna(test_le['Electrical'].mode()[0], inplace=True)

garage_cols = ['GarageType', 'GarageYrBlt', 'GarageFinish', 'GarageQual', 'GarageCond']
bsmt_cols = ['BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2']

In [199]:
for col in garage_cols + bsmt_cols:
    if test_le[col].dtype == 'object':
        test_le[col].fillna('None', inplace=True)
    else:
        test_le[col].fillna(0, inplace=True)

print(test_le.isnull().sum())

Id               0
MSSubClass       0
MSZoning         4
LotFrontage      0
LotArea          0
                ..
MiscVal          0
MoSold           0
YrSold           0
SaleType         1
SaleCondition    0
Length: 74, dtype: int64


In [200]:
test_le_id = test_le['Id']
test_le.drop('Id', axis=1, inplace=True)
test_le = test_le.reindex(columns=X.columns, fill_value=0)

In [201]:
cat_cols = test_le.select_dtypes(include='object').columns

for col in cat_cols:
    test_le[col] = le.fit_transform(test_le[col])

In [202]:
test_le = test_le.reindex(columns=X.columns, fill_value=0)

In [203]:
test_le = test_le.fillna(X.mean())

In [204]:
print(test_le.isnull().sum())

Id               0
MSSubClass       0
MSZoning         0
LotFrontage      0
LotArea          0
                ..
MiscVal          0
MoSold           0
YrSold           0
SaleType         0
SaleCondition    0
Length: 74, dtype: int64


In [206]:
test_preds = model.predict(test_le)

In [209]:
submission = pd.DataFrame({
    'Id': test_id,
    'SalePrice': test_preds
})

submission.to_csv("submission.csv", index =False)
print('Submission file is ready')

Submission file is ready
